In [1]:
%load_ext autoreload
%autoreload 2

In [2]:

from pyspark_data_provenance import (
    data_provenance_enabled, 
    build_data_provenance_session
)

In [3]:
from pyspark.sql import SparkSession


spark = (
    # SparkSession
    # .builder
    build_data_provenance_session()
    .appName("data-provenance-notebook")
    .getOrCreate()
)


26/04/21 19:48:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
print(spark.conf.get("spark.provenance.enabled", "false"))

with data_provenance_enabled(spark):
    print(spark.conf.get("spark.provenance.enabled", "false"))

print(spark.conf.get("spark.provenance.enabled", "false"))

false
true
false


In [5]:
from datetime import date

df = spark.createDataFrame([
    ("A", date(2026, 1, 15), 10.0, 90),
    ("A", date(2026, 1, 16), 10.0, 120),
    ("A", date(2026, 1, 17), 5.0, 300),
    ("B", date(2026, 1, 15), 100.0, 20),
    ("B", date(2026, 1, 16), 100.0, 30),
    ("C", date(2026, 1, 17), 80.0, 60),
    ("F", date(2026, 1, 16), 50.0, 70)
], ["product", "date", "price", "quantity"]
)
df.printSchema()

df.toPandas()

root
 |-- product: string (nullable = true)
 |-- date: date (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: long (nullable = true)



,product,date,price,quantity
0,A,2026-01-15,10.0,90
1,A,2026-01-16,10.0,120
2,A,2026-01-17,5.0,300
3,B,2026-01-15,100.0,20
4,B,2026-01-16,100.0,30
5,C,2026-01-17,80.0,60
6,F,2026-01-16,50.0,70


In [6]:
df2 = spark.createDataFrame([
    ("A", "Bike"),
    ("B", "Handball"),
    ("C", "Bike"),
    ("D", "Handball"),
    ("E", "Running")
],["product","labell"]
)

df2.printSchema()
df2.toPandas()



root
 |-- product: string (nullable = true)
 |-- labell: string (nullable = true)



,product,labell
0,A,Bike
1,B,Handball
2,C,Bike
3,D,Handball
4,E,Running


## Tests 

### SQL

In [7]:
df.createOrReplaceTempView("sales")
df2.createOrReplaceTempView("category")
spark.sql("select * from sales s join category c on s.product=c.product").toPandas()

,product,date,price,quantity,product,labell
0,A,2026-01-15,10.0,90,A,Bike
1,A,2026-01-16,10.0,120,A,Bike
2,A,2026-01-17,5.0,300,A,Bike
3,B,2026-01-15,100.0,20,B,Handball
4,B,2026-01-16,100.0,30,B,Handball
5,C,2026-01-17,80.0,60,C,Bike


In [8]:
df.createOrReplaceTempView("sales")
df2.createOrReplaceTempView("category")
with data_provenance_enabled(spark):
    res = spark.sql("select * from sales s join category c on s.product=c.product").toPandas()

res

,product,date,price,quantity,product,labell,_provenance_tag_select
0,A,2026-01-15,10.0,90,A,Bike,0
1,A,2026-01-16,10.0,120,A,Bike,1
2,A,2026-01-17,5.0,300,A,Bike,2
3,B,2026-01-15,100.0,20,B,Handball,3
4,B,2026-01-16,100.0,30,B,Handball,4
5,C,2026-01-17,80.0,60,C,Bike,5


### Spark

In [9]:
df5 = df.select("product")

df5.toPandas()


,product
0,A
1,A
2,A
3,B
4,B
5,C
6,F


In [18]:

with data_provenance_enabled(spark):
    df7 = df.join(df2,"product").select("product")

df7.toPandas()

,product,_provenance_tag_select
0,A,0
1,A,1
2,A,2
3,B,3
4,B,4
5,C,5


In [11]:
df3 = df.select("*").filter("price > 10")
df3.toPandas()
#df3.explain("extended")

,product,date,price,quantity
0,B,2026-01-15,100.0,20
1,B,2026-01-16,100.0,30
2,C,2026-01-17,80.0,60
3,F,2026-01-16,50.0,70


In [12]:
#df.createOrReplaceTempView("sales")

#with data_provenance_enabled(spark):
    #res = spark.sql("select * from sales where price>10").toPandas()

#res

In [13]:
df4 = df.select("*").filter("price > 10").sort("price").join(df2,"product")
df4.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [product])
:- Sort [price#2 ASC NULLS FIRST], true
:  +- Filter (price#2 > cast(10 as double))
:     +- Project [product#0, date#1, price#2, quantity#3L]
:        +- LogicalRDD [product#0, date#1, price#2, quantity#3L], false
+- LogicalRDD [product#4, labell#5], false

== Analyzed Logical Plan ==
product: string, date: date, price: double, quantity: bigint, labell: string
Project [product#0, date#1, price#2, quantity#3L, labell#5]
+- Join Inner, (product#0 = product#4)
   :- Sort [price#2 ASC NULLS FIRST], true
   :  +- Filter (price#2 > cast(10 as double))
   :     +- Project [product#0, date#1, price#2, quantity#3L]
   :        +- LogicalRDD [product#0, date#1, price#2, quantity#3L], false
   +- LogicalRDD [product#4, labell#5], false

== Optimized Logical Plan ==
Project [product#0, date#1, price#2, quantity#3L, labell#5]
+- Join Inner, (product#0 = product#4)
   :- Filter ((isnotnull(price#2) AND (price#2 > 10.0)) AND isnotnull(produ

In [14]:
df6 = df.select("*").join(df2, "product", "outer")
df6.toPandas()

,product,date,price,quantity,labell
0,A,2026-01-15,10.0,90.0,Bike
1,A,2026-01-16,10.0,120.0,Bike
2,A,2026-01-17,5.0,300.0,Bike
3,B,2026-01-15,100.0,20.0,Handball
4,B,2026-01-16,100.0,30.0,Handball
5,C,2026-01-17,80.0,60.0,Bike
6,D,None,NaN,NaN,Handball
7,E,None,NaN,NaN,Running
8,F,2026-01-16,50.0,70.0,NaN


In [15]:
with data_provenance_enabled(spark) :
    dfy = df.select("*").join(df2, "product", "outer")
    dfy.toPandas()

In [16]:
df7 = df.groupBy("product").agg({"sales": "sum"}).sort("product")
df7.toPandas()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `sales` cannot be resolved. Did you mean one of the following? [`product`, `date`, `price`, `quantity`]. SQLSTATE: 42703

In [ ]:
df8 = df.withColumn("revenue", df["sales"] * df["price"]) \
        .groupBy("product") \
        .agg({"revenue": "sum"}) \
        .sort("product")

df8.toPandas()
df8.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 2
   +- *(3) Sort [product#83 ASC NULLS FIRST], true, 0
      +- AQEShuffleRead coalesced
         +- ShuffleQueryStage 1
            +- Exchange rangepartitioning(product#83 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=2773]
               +- *(2) HashAggregate(keys=[product#83], functions=[sum(revenue#106)])
                  +- AQEShuffleRead coalesced
                     +- ShuffleQueryStage 0
                        +- Exchange hashpartitioning(product#83, 200), ENSURE_REQUIREMENTS, [plan_id=2749]
                           +- *(1) HashAggregate(keys=[product#83], functions=[partial_sum(revenue#106)])
                              +- *(1) Project [product#83, (cast(sales#86L as double) * price#85) AS revenue#106]
                                 +- *(1) Scan ExistingRDD[product#83,date#84,price#85,sales#86L]
+- == Initial Plan ==
   Sort [product#83 ASC NULLS FIRST], true, 0
   